# Phase 4: behavior on Qwen2.5-7B-Instruct (bf16, A100 or L4)

**Before running:** `Runtime → Change runtime type → A100 GPU` (L4 also works), then `Runtime → Run all`.
Clone access works the same as the pilot (`GH_TOKEN` secret, or a public repo).

1. **GPU check**: stops right away if the GPU can't run the 7B in bf16.
2. **Smoke test**: every script on 4 questions with the real 7B, into a throwaway folder (a couple of minutes).
   If anything is broken it fails here, before the expensive run.
3. **Full run**: all 482 questions × 13 conditions baseline, then every kept question × 13 × 3 templates.
   Checkpointed to Drive every 512 prompts: **if Colab disconnects, just `Run all` again** and it resumes.
4. **Stats** and the **free-text reality check**.
5. **Summary** to paste back to Claude, then the runtime **disconnects itself** so it stops using compute units
   (set `AUTO_DISCONNECT = False` below to keep it).

Every step raises an error if its script fails, so `Run all` stops at the first problem.
Outputs: `MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct/`.

In [ ]:
CFG = "configs/colab.yaml"   # bf16 7B. Use this same config for every later 7B phase.
AUTO_DISCONNECT = True

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None
url = f"https://{token}@github.com/GTRAN7/vnpronouns.git" if token else "https://github.com/GTRAN7/vnpronouns.git"
if not os.path.exists('/content/vnpronouns'):
    subprocess.run(['git', 'clone', '-q', url, '/content/vnpronouns'], check=True)
else:
    subprocess.run(['git', '-C', '/content/vnpronouns', 'pull', '-q'], check=True)
%cd /content/vnpronouns
!git log --oneline -1
!pip install -q -e .

def run(cmd, log, env=None):
    """Run a shell command, stream + log its output, and raise if it fails (so Run all stops)."""
    print(f'$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1', 'TQDM_MININTERVAL': '30', **(env or {})})
    with open(log, 'a', encoding='utf-8') as f:
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
    if p.wait() != 0:
        raise RuntimeError(f'FAILED (exit {p.returncode}): {cmd}  -- see {log}')

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> A100 (or L4)'
name = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f'{name}, {mem:.0f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported() and mem >= 22, f'{name} cannot hold the 7B in bf16; switch to A100/L4'

## 2. Smoke test: every script on 4 questions with the real 7B (throwaway folder, not Drive)

In [ ]:
import time
SMOKE = {'VNP_OUTPUT_DIR': '/content/smoke'}
!rm -rf /content/smoke
t0 = time.time()
run(f'python -m vnpronouns.run_behavior --config {CFG} --pool 4 --no-filter', '/content/log_smoke.txt', SMOKE)
run(f'python -m vnpronouns.stats --config {CFG} --n-boot 500', '/content/log_smoke.txt', SMOKE)
run(f'python -m vnpronouns.freetext --config {CFG} --n-questions 2', '/content/log_smoke.txt', SMOKE)
print(f'\nSMOKE TEST PASSED in {time.time() - t0:.0f}s')

## 3. Full run: baseline filter (all questions) + flip rates on every kept question
The long step. Re-run after a disconnect to resume.

In [ ]:
run(f'python -m vnpronouns.run_behavior --config {CFG}', '/content/log_behavior.txt')

## 4. Statistics: flip rates with 95% bootstrap CIs, McNemar (Bonferroni ×3), per-pair / gender / template breakdowns

In [ ]:
!rm -f /content/log_stats.txt
run(f'python -m vnpronouns.stats --config {CFG}', '/content/log_stats.txt')

## 5. Reality check: free-text replies on 50 questions × 13 conditions (flat template)

In [ ]:
!rm -f /content/log_freetext.txt
run(f'python -m vnpronouns.freetext --config {CFG} --n-questions 50', '/content/log_freetext.txt')

## 6. Summary to paste back to Claude (then the runtime disconnects if AUTO_DISCONNECT)

In [ ]:
import re
def clean(path, last_run_only=False):
    lines = open(path, encoding='utf-8').read().replace('\r', '\n').splitlines()
    if last_run_only:  # the behavior log is appended across resumes; show the final run
        starts = [i for i, l in enumerate(lines) if l.startswith('baseline:')]
        lines = lines[starts[-1]:] if starts else lines
    keep = [l.rstrip() for l in lines
            if not re.search(r'it/s\]|s/it\]|Warning|warn\(|^\s*$|Loading|Fetching|Downloading|%\||^  chunk |^\$ ', l)]
    return '\n'.join(keep)
print(f'GPU: {torch.cuda.get_device_name(0)}  config: {CFG}')
for p, last in [('/content/log_behavior.txt', True), ('/content/log_stats.txt', False), ('/content/log_freetext.txt', False)]:
    print(f'===== {p.split("/")[-1]} =====')
    print(clean(p, last))
!ls -la /content/drive/MyDrive/vnpronouns/outputs/Qwen2.5-7B-Instruct/

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    print('Done. Disconnecting the runtime to save compute units (outputs above stay visible).')
    runtime.unassign()